In [5]:
# Import Libraries
import pandas as pd
import numpy as np
import os
import shutil
from google.colab import files

# Create folders
os.makedirs("Original_Data", exist_ok=True)
os.makedirs("Cleaned_Data", exist_ok=True)

# Upload Dataset Files
uploaded = files.upload()

# Save Original Copies
for filename, content in uploaded.items():
    with open(os.path.join("Original_Data", filename), "wb") as f:
        f.write(content)

# Read All Files
data = {}

for filename in uploaded.keys():

    if filename.endswith(".xlsx"):
        df = pd.read_excel(filename)

    elif filename.endswith(".csv"):
        df = pd.read_csv(filename)

    else:
        continue

    name = filename.split(".")[0]
    data[name] = df

# Display Dataset Information
for name, df in data.items():

    print("\n" + "=" * 40)
    print("Dataset:", name)
    print("Rows and Columns:", df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

print("\nAll datasets loaded successfully!")

KeyboardInterrupt: 

In [ ]:
import os

print("Uploaded files:")
print(os.listdir('/content'))

In [ ]:
import pandas as pd
import os

files = [
    '01_Customers.csv',
    '02_Products.csv',
    '03_Suppliers.csv',
    '04_Warehouses.csv',
    '05_Orders.csv',
    '06_Order_Items.csv',
    '07_Reviews.csv',
    '08_Inventory.csv'
]

for file in files:
    path = f'/content/{file}'
    df = pd.read_csv(path)

    print("\n" + "="*50)
    print("Dataset:", file)
    print("Shape:", df.shape)
    print("Missing Values:", df.isnull().sum().sum())
    print("Duplicate Rows:", df.duplicated().sum())

In [ ]:

import pandas as pd
import os

files = [
    '01_Customers.csv',
    '02_Products.csv',
    '03_Suppliers.csv',
    '04_Warehouses.csv',
    '05_Orders.csv',
    '06_Order_Items.csv',
    '07_Reviews.csv',
    '08_Inventory.csv'
]

for file in files:
    df = pd.read_csv(f'/content/{file}')

    print("\n" + "="*60)
    print("FILE:", file)
    print("COLUMNS:", list(df.columns))
    print("\nDATA TYPES:")
    print(df.dtypes)
    print("\nSAMPLE DATA:")
    print(df.head(3).to_string(index=False))


In [ ]:

import pandas as pd
import os
import shutil

# Create folders
original_folder = '/content/Original_Data'
clean_folder = '/content/Cleaned_Data'

os.makedirs(original_folder, exist_ok=True)
os.makedirs(clean_folder, exist_ok=True)

files = [
    '01_Customers.csv',
    '02_Products.csv',
    '03_Suppliers.csv',
    '04_Warehouses.csv',
    '05_Orders.csv',
    '06_Order_Items.csv',
    '07_Reviews.csv',
    '08_Inventory.csv'
]

cleaning_log = []
cleaned_data = {}

for file in files:

    source = f'/content/{file}'
    df = pd.read_csv(source)

    # Save original copy without modifications
    original_path = f'{original_folder}/{file}'
    if not os.path.exists(original_path):
        shutil.copy2(source, original_path)

    original_rows = len(df)
    original_missing = int(df.isnull().sum().sum())
    original_duplicates = int(df.duplicated().sum())

    # Remove exact duplicate rows
    df = df.drop_duplicates().copy()

    removed_duplicates = original_rows - len(df)

    # Standardize text columns
    text_columns = df.select_dtypes(include='object').columns

    for col in text_columns:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    # Convert date columns
    for col in df.columns:
        if 'date' in col.lower():
            df[col] = pd.to_datetime(
                df[col], errors='coerce'
            )

    # Save cleaned dataset
    output_name = file.replace('.csv', '_clean.csv')
    df.to_csv(
        f'{clean_folder}/{output_name}',
        index=False
    )

    cleaned_data[file] = df

    cleaning_log.append({
        'Dataset': file,
        'Original Rows': original_rows,
        'Duplicates Found': original_duplicates,
        'Duplicates Removed': removed_duplicates,
        'Missing Values Before': original_missing,
        'Missing Values After': int(df.isnull().sum().sum()),
        'Final Rows': len(df),
        'Action': 'Removed exact duplicates, trimmed text, standardized dates'
    })

# Save cleaning log
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(
    '/content/Cleaning_Log.csv',
    index=False
)

print("DATA CLEANING COMPLETED")
print(log_df.to_string(index=False))
print("\nCleaned files saved in:", clean_folder)
print("Cleaning log saved successfully.")


In [ ]:

import pandas as pd
import os

folder = '/content/Cleaned_Data'

data = {
    file: pd.read_csv(f'{folder}/{file}_clean.csv')
    for file in [
        '01_Customers',
        '02_Products',
        '03_Suppliers',
        '04_Warehouses',
        '05_Orders',
        '06_Order_Items',
        '07_Reviews',
        '08_Inventory'
    ]
}

# 1. Missing values by column
print("=" * 60)
print("MISSING VALUES DETAILS")

for name, df in data.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    if len(missing):
        print(missing.to_string())
    else:
        print("No missing values")

# 2. Check primary key duplicates
primary_keys = {
    '01_Customers': 'customer_id',
    '02_Products': 'product_id',
    '03_Suppliers': 'supplier_id',
    '04_Warehouses': 'warehouse_id',
    '05_Orders': 'order_id',
    '06_Order_Items': 'order_item_id',
    '07_Reviews': 'review_id',
    '08_Inventory': 'inventory_id'
}

print("\n" + "=" * 60)
print("PRIMARY KEY CHECK")

for name, key in primary_keys.items():
    df = data[name]
    print(name, "duplicate IDs:", df[key].duplicated().sum())

# 3. Check relationships between datasets
relationships = [
    ('05_Orders', 'customer_id', '01_Customers', 'customer_id'),
    ('05_Orders', 'warehouse_id', '04_Warehouses', 'warehouse_id'),
    ('06_Order_Items', 'order_id', '05_Orders', 'order_id'),
    ('06_Order_Items', 'product_id', '02_Products', 'product_id'),
    ('02_Products', 'supplier_id', '03_Suppliers', 'supplier_id'),
    ('07_Reviews', 'order_id', '05_Orders', 'order_id'),
    ('08_Inventory', 'warehouse_id', '04_Warehouses', 'warehouse_id'),
    ('08_Inventory', 'product_id', '02_Products', 'product_id')
]

print("\n" + "=" * 60)
print("RELATIONSHIP CHECK")

for child, child_col, parent, parent_col in relationships:
    child_values = data[child][child_col].dropna()
    parent_values = set(data[parent][parent_col].dropna())

    unmatched = (~child_values.isin(parent_values)).sum()

    print(f"{child}.{child_col} -> {parent}.{parent_col}")
    print("Unmatched values:", unmatched)

print("\nINSPECTION COMPLETED")


In [ ]:

import pandas as pd

folder = '/content/Cleaned_Data'

customers = pd.read_csv(f'{folder}/01_Customers_clean.csv')
orders = pd.read_csv(f'{folder}/05_Orders_clean.csv')
items = pd.read_csv(f'{folder}/06_Order_Items_clean.csv')
reviews = pd.read_csv(f'{folder}/07_Reviews_clean.csv')
inventory = pd.read_csv(f'{folder}/08_Inventory_clean.csv')
products = pd.read_csv(f'{folder}/02_Products_clean.csv')

print("1. MISSING CUSTOMER IDs")
print(orders[orders['customer_id'].isna()].to_string(index=False))

print("\n2. DUPLICATE REVIEW IDs")
print(reviews[reviews['review_id'].duplicated(keep=False)].to_string(index=False))

print("\n3. UNMATCHED PRODUCT IDs")
print(items[~items['product_id'].isin(products['product_id'])].to_string(index=False))

print("\n4. REVIEWS WITH INVALID ORDER IDs")
print(reviews[~reviews['order_id'].isin(orders['order_id'])].to_string(index=False))

print("\n5. MISSING CUSTOMER DATA")
print(customers[customers[['age','city']].isna().any(axis=1)].to_string(index=False))

print("\n6. MISSING ORDER DATA")
print(orders[orders[['campaign','delivery_days']].isna().any(axis=1)].head(15).to_string(index=False))

print("\n7. DATA DISTRIBUTIONS")
print("\nCustomer ages:")
print(customers['age'].describe())

print("\nOrder status and delivery days:")
print(orders.groupby('order_status')['delivery_days'].agg(['count','mean','median']))

print("\nQuantity:")
print(items['quantity'].describe())

print("\nDiscount:")
print(items['discount_pct'].describe())

print("\nReview ratings:")
print(reviews['rating'].value_counts(dropna=False).sort_index())

print("\nInventory stock:")
print(inventory['stock_on_hand'].describe())


In [ ]:

import pandas as pd

folder = '/content/Cleaned_Data'

items = pd.read_csv(f'{folder}/06_Order_Items_clean.csv')
reviews = pd.read_csv(f'{folder}/07_Reviews_clean.csv')
inventory = pd.read_csv(f'{folder}/08_Inventory_clean.csv')
orders = pd.read_csv(f'{folder}/05_Orders_clean.csv')

print("1. ZERO OR INVALID QUANTITIES")
print(items[items['quantity'] <= 0].to_string(index=False))

print("\n2. HIGH DISCOUNTS (ABOVE 50%)")
print(items[items['discount_pct'] > 0.5].to_string(index=False))

print("\n3. INVALID RATINGS")
print(reviews[
    (reviews['rating'] < 1) |
    (reviews['rating'] > 5)
].to_string(index=False))

print("\n4. NEGATIVE INVENTORY")
print(inventory[
    inventory['stock_on_hand'] < 0
].to_string(index=False))

print("\n5. MISSING DELIVERY DAYS BY STATUS")
print(
    orders.groupby('order_status')['delivery_days']
    .agg(['count', 'size', 'median'])
)

print("\nINSPECTION COMPLETED")


In [ ]:

import pandas as pd
import numpy as np
import os

base = '/content/Cleaned_Data'
quarantine = '/content/Quarantined_Data'

os.makedirs(quarantine, exist_ok=True)

# Load cleaned datasets
customers = pd.read_csv(f'{base}/01_Customers_clean.csv')
products = pd.read_csv(f'{base}/02_Products_clean.csv')
suppliers = pd.read_csv(f'{base}/03_Suppliers_clean.csv')
warehouses = pd.read_csv(f'{base}/04_Warehouses_clean.csv')
orders = pd.read_csv(f'{base}/05_Orders_clean.csv')
items = pd.read_csv(f'{base}/06_Order_Items_clean.csv')
reviews = pd.read_csv(f'{base}/07_Reviews_clean.csv')
inventory = pd.read_csv(f'{base}/08_Inventory_clean.csv')

log = []

def record(dataset, column, action, reason, count):
    log.append({
        'Dataset': dataset,
        'Column': column,
        'Action': action,
        'Reason': reason,
        'Affected Rows': count
    })

# 1. Customers
age_median = customers['age'].median()
n = customers['age'].isna().sum()
customers['age'] = customers['age'].fillna(age_median)
record('Customers', 'age', 'Median imputation',
       'Used median age', n)

n = customers['city'].isna().sum()
customers['city'] = customers['city'].fillna('Unknown')
record('Customers', 'city', 'Filled with Unknown',
       'City was not provided', n)

# 2. Orders
n = orders['campaign'].isna().sum()
orders['campaign'] = orders['campaign'].fillna('Unknown')
record('Orders', 'campaign', 'Filled with Unknown',
       'Campaign was not recorded', n)

n = orders['delivery_days'].isna().sum()
cancelled = orders['order_status'].eq('Cancelled')

for status in orders['order_status'].dropna().unique():
    if status == 'Cancelled':
        continue

    mask = (
        orders['order_status'].eq(status) &
        orders['delivery_days'].isna()
    )

    median = orders.loc[
        orders['order_status'].eq(status), 'delivery_days'
    ].median()

    orders.loc[mask, 'delivery_days'] = median

record('Orders', 'delivery_days',
       'Median by order status',
       'Cancelled orders kept missing; other statuses imputed', n)

# 3. Order Items
n = items['quantity'].isna().sum()
items['quantity'] = items['quantity'].fillna(
    items['quantity'].median()
)
record('Order Items', 'quantity', 'Median imputation',
       'Filled missing quantities', n)

n = items['discount_pct'].isna().sum()
items['discount_pct'] = items['discount_pct'].fillna(
    items['discount_pct'].median()
)
record('Order Items', 'discount_pct', 'Median imputation',
       'Filled missing discounts', n)

# Quarantine invalid order items
valid_products = set(products['product_id'])

bad_items = items[
    (items['quantity'] <= 0) |
    (~items['product_id'].isin(valid_products))
].copy()

bad_items.to_csv(
    f'{quarantine}/Invalid_Order_Items.csv', index=False
)

items = items.drop(index=bad_items.index).copy()

record('Order Items', 'records', 'Quarantined',
       'Zero quantity or unmatched product ID', len(bad_items))

# Flag unusual discounts without deleting them
high_discount = items['discount_pct'] > 0.5

record('Order Items', 'discount_pct', 'Flagged',
       'Discount greater than 50%', high_discount.sum())

# 4. Reviews
bad_rating = reviews['rating'].notna() & (
    (reviews['rating'] < 1) | (reviews['rating'] > 5)
)

n = bad_rating.sum()
reviews.loc[bad_rating, 'rating'] = np.nan

record('Reviews', 'rating', 'Set to missing',
       'Rating outside valid range 1-5', n)

n = reviews['review_text'].isna().sum()
reviews['review_text'] = reviews['review_text'].fillna(
    'Not provided'
)
record('Reviews', 'review_text', 'Filled with Not provided',
       'Review text missing', n)

# Quarantine invalid review references
valid_orders = set(orders['order_id'])

bad_reviews = reviews[
    ~reviews['order_id'].isin(valid_orders)
].copy()

bad_reviews.to_csv(
    f'{quarantine}/Invalid_Reviews.csv', index=False
)

reviews = reviews[
    reviews['order_id'].isin(valid_orders)
].copy()

record('Reviews', 'order_id', 'Quarantined',
       'Review refers to a nonexistent order', len(bad_reviews))

# Resolve duplicate review IDs, preferring valid unique records
duplicate_reviews = reviews[
    reviews['review_id'].duplicated(keep=False)
].copy()

reviews = reviews.drop_duplicates(
    subset='review_id', keep='first'
).copy()

record('Reviews', 'review_id', 'Removed duplicate IDs',
       'Kept one record per review ID',
       len(duplicate_reviews))

# 5. Inventory
bad_inventory = inventory[
    inventory['stock_on_hand'] < 0
].copy()

bad_inventory.to_csv(
    f'{quarantine}/Invalid_Inventory.csv', index=False
)

inventory = inventory[
    inventory['stock_on_hand'] >= 0
].copy()

record('Inventory', 'stock_on_hand', 'Quarantined',
       'Negative inventory values need verification',
       len(bad_inventory))

# 6. Save cleaned individual datasets
datasets = {
    '01_Customers_clean.csv': customers,
    '02_Products_clean.csv': products,
    '03_Suppliers_clean.csv': suppliers,
    '04_Warehouses_clean.csv': warehouses,
    '05_Orders_clean.csv': orders,
    '06_Order_Items_clean.csv': items,
    '07_Reviews_clean.csv': reviews,
    '08_Inventory_clean.csv': inventory
}

for filename, df in datasets.items():
    df.to_csv(f'{base}/{filename}', index=False)

# 7. Aggregate reviews to one row per order
review_summary = reviews.groupby('order_id').agg(
    review_count=('review_id', 'count'),
    average_rating=('rating', 'mean'),
    review_text=('review_text',
                 lambda x: ' | '.join(x.astype(str)))
).reset_index()

# 8. Merge datasets at order-item level
merged = items.merge(
    orders, on='order_id', how='left',
    validate='many_to_one'
)

merged = merged.merge(
    customers, on='customer_id', how='left',
    validate='many_to_one'
)

merged = merged.merge(
    products, on='product_id', how='left',
    validate='many_to_one'
)

merged = merged.merge(
    suppliers, on='supplier_id', how='left',
    validate='many_to_one'
)

merged = merged.merge(
    warehouses, on='warehouse_id', how='left',
    validate='many_to_one'
)

merged = merged.merge(
    review_summary, on='order_id', how='left',
    validate='many_to_one'
)

# Save merged dataset
merged.to_csv(
    '/content/Cleaned_Merged_Data.csv', index=False
)

# Save inventory separately because it has warehouse-product grain
inventory.to_csv(
    '/content/Cleaned_Inventory.csv', index=False
)

# Save cleaning log
log_df = pd.DataFrame(log)
log_df.to_csv('/content/Cleaning_Log.csv', index=False)

print("FINAL CLEANING AND MERGING COMPLETED")
print("Merged dataset shape:", merged.shape)
print("Inventory shape:", inventory.shape)
print("Quarantined Order Items:", len(bad_items))
print("Quarantined Reviews:", len(bad_reviews))
print("Quarantined Inventory:", len(bad_inventory))
print("\nCleaning Log:")
print(log_df.to_string(index=False))

print("\nFILES CREATED:")
print("Cleaned_Merged_Data.csv")
print("Cleaned_Inventory.csv")
print("Cleaning_Log.csv")
print("Quarantined_Data folder")


In [ ]:

import pandas as pd
import os

source = '/content/Original_Data/08_Inventory.csv'

inventory = pd.read_csv(source)

# Remove exact duplicate rows
inventory = inventory.drop_duplicates().copy()

# Isolate negative stock only
negative_stock = inventory[
    inventory['stock_on_hand'] < 0
].copy()

negative_stock.to_csv(
    '/content/Quarantined_Data/Invalid_Inventory.csv',
    index=False
)

# Keep zero, positive and missing stock values
clean_inventory = inventory[
    (inventory['stock_on_hand'] >= 0) |
    (inventory['stock_on_hand'].isna())
].copy()

clean_inventory.to_csv(
    '/content/Cleaned_Inventory.csv',
    index=False
)

clean_inventory.to_csv(
    '/content/Cleaned_Data/08_Inventory_clean.csv',
    index=False
)

print("INVENTORY CORRECTION COMPLETED")
print("Original rows:", len(inventory))
print("Negative rows isolated:", len(negative_stock))
print("Final inventory rows:", len(clean_inventory))
print("Missing stock values kept:",
      clean_inventory['stock_on_hand'].isna().sum())


In [ ]:

import pandas as pd
import os
import zipfile

# Load final files
merged = pd.read_csv('/content/Cleaned_Merged_Data.csv')
inventory = pd.read_csv('/content/Cleaned_Inventory.csv')
log = pd.read_csv('/content/Cleaning_Log.csv')

# Correct the review duplicate documentation
mask = (
    (log['Dataset'] == 'Reviews') &
    (log['Column'] == 'review_id')
)

log.loc[mask, 'Action'] = 'Duplicate ID reviewed'
log.loc[mask, 'Reason'] = (
    'Two rows shared R00637; invalid order reference quarantined'
)
log.loc[mask, 'Affected Rows'] = 2

log.to_csv('/content/Cleaning_Log.csv', index=False)

# Validate merged dataset
print("FINAL QUALITY CHECK")
print("=" * 45)

print("Merged rows:", len(merged))
print("Merged columns:", len(merged.columns))
print("Duplicate order item IDs:",
      merged['order_item_id'].duplicated().sum())

print("Inventory rows:", len(inventory))
print("Inventory negative values:",
      (inventory['stock_on_hand'] < 0).sum())

print("Inventory missing values:",
      inventory['stock_on_hand'].isna().sum())

print("Missing values in merged data:",
      merged.isnull().sum().sum())

# Check required files
required = [
    '/content/Cleaned_Merged_Data.csv',
    '/content/Cleaned_Inventory.csv',
    '/content/Cleaning_Log.csv',
    '/content/Original_Data',
    '/content/Cleaned_Data',
    '/content/Quarantined_Data'
]

print("\nREQUIRED FILES CHECK")
for path in required:
    print(os.path.basename(path), os.path.exists(path))

# Create README
readme = """
DATA ENGINEERING PROJECT - FINAL DELIVERY

1. Cleaned_Merged_Data.csv
   Main merged dataset at order-item level.

2. Cleaned_Inventory.csv
   Inventory dataset kept separately.

3. Cleaning_Log.csv
   Documentation of cleaning actions.

4. Original_Data/
   Original source files preserved without modification.

5. Cleaned_Data/
   Cleaned individual datasets.

6. Quarantined_Data/
   Records isolated for further verification.

Note:
Missing customer IDs and missing delivery days were
preserved where reliable replacement values were unavailable.
Inventory with missing stock values was retained.
"""

with open('/content/README.txt', 'w') as f:
    f.write(readme)

# Package all deliverables
zip_path = '/content/Data_Engineering_Delivery.zip'

with zipfile.ZipFile(zip_path, 'w',
                     zipfile.ZIP_DEFLATED) as z:
    for path in required + [
        '/content/README.txt'
    ]:
        if os.path.isfile(path):
            z.write(path, os.path.basename(path))
        elif os.path.isdir(path):
            for root, dirs, files in os.walk(path):
                for file in files:
                    full = os.path.join(root, file)
                    relative = os.path.relpath(
                        full, '/content'
                    )
                    z.write(full, relative)

print("\nDELIVERY PACKAGE CREATED")
print(zip_path)
print("FINAL CHECK COMPLETED")


In [7]:
from google.colab import files
files.download('/content/Data_Engineering_Delivery.zip')

FileNotFoundError: Cannot find file: /content/Data_Engineering_Delivery.zip

In [8]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect('project.db')

files_mapping = {
    '01_Customers': 'customers',
    '02_Products': 'products',
    '03_Suppliers': 'suppliers',
    '04_Warehouses': 'warehouses',
    '05_Orders': 'orders',
    '06_Order_Items': 'order_items',
    '07_Reviews': 'reviews',
    '08_Inventory': 'inventory'
}

for file_prefix, table_name in files_mapping.items():
    for filename in os.listdir('/content/'):
        if filename.startswith(file_prefix) and filename.endswith('.csv'):
            df = pd.read_csv(f'/content/{filename}')
            df.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"تم تجهيز: {table_name}")

تم تجهيز: customers
تم تجهيز: products
تم تجهيز: suppliers
تم تجهيز: warehouses
تم تجهيز: orders
تم تجهيز: order_items
تم تجهيز: reviews
تم تجهيز: inventory


In [9]:
query1 = """
SELECT
    (SELECT COUNT(*) FROM customers) AS Total_Customers,
    (SELECT COUNT(*) FROM orders) AS Total_Orders,
    (SELECT ROUND(SUM(quantity * unit_price), 2) FROM order_items) AS Total_Sales_SAR,
    (SELECT COUNT(*) FROM reviews WHERE rating <= 2) AS Total_Complaints
FROM (SELECT 1) AS dummy_table;
"""
result1 = pd.read_sql_query(query1, conn)
print("--- 1. المؤشرات الأساسية ---")
print(result1)
result1.to_csv('1_indicators.csv', index=False)

--- 1. المؤشرات الأساسية ---
   Total_Customers  Total_Orders  Total_Sales_SAR  Total_Complaints
0              503          1400       1000502.61               142


In [10]:
query2 = """
SELECT c.city, COUNT(r.review_id) AS Complaint_Count
FROM reviews r
JOIN orders o ON r.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE r.rating <= 2
GROUP BY c.city
ORDER BY Complaint_Count DESC
LIMIT 5;
"""
result2 = pd.read_sql_query(query2, conn)
print("--- 2. المدن الأكثر شكوى ---")
print(result2)
result2.to_csv('2_complaints_by_city.csv', index=False)

--- 2. المدن الأكثر شكوى ---
      city  Complaint_Count
0   Jubail               15
1    Tabuk               13
2   Riyadh               12
3   Najran               12
4  Majmaah               12


In [12]:
query3 = """
SELECT
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS Total_Cost_of_Complaints_SAR
FROM reviews r
JOIN orders o ON r.order_id = o.order_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE r.rating <= 2;
"""
result3 = pd.read_sql_query(query3, conn)
print("--- 3. تكلفة المشكلة بالريال ---")
print(result3)
result3.to_csv('3_complaints_cost.csv', index=False)

--- 3. تكلفة المشكلة بالريال ---
   Total_Cost_of_Complaints_SAR
0                     107636.34


In [13]:
query4 = """
SELECT
    o.order_status,
    COUNT(r.review_id) AS Number_of_Reviews,
    ROUND(AVG(r.rating), 2) AS Average_Rating
FROM orders o
JOIN reviews r ON o.order_id = r.order_id
GROUP BY o.order_status
ORDER BY Average_Rating ASC;
"""
result4 = pd.read_sql_query(query4, conn)
print("--- 4. علاقة حالة الطلب بالتقييم ---")
print(result4)
result4.to_csv('4_delay_vs_rating.csv', index=False)

--- 4. علاقة حالة الطلب بالتقييم ---
  order_status  Number_of_Reviews  Average_Rating
0     Returned                 89            3.76
1    Delivered                764            3.83


In [14]:
!zip results.zip *.csv
print("تم التجميع بنجاح!")

  adding: 01_Customers.csv (deflated 81%)
  adding: 02_Products.csv (deflated 50%)
  adding: 03_Suppliers.csv (deflated 53%)
  adding: 04_Warehouses.csv (deflated 37%)
  adding: 05_Orders.csv (deflated 81%)
  adding: 06_Order_Items.csv (deflated 79%)
  adding: 07_Reviews.csv (deflated 86%)
  adding: 08_Inventory.csv (deflated 76%)
  adding: 1_indicators.csv (deflated 19%)
  adding: 2_complaints_by_city.csv (deflated 6%)
  adding: 3_complaints_cost.csv (stored 0%)
  adding: 4_delay_vs_rating.csv (deflated 4%)
تم التجميع بنجاح!
